For Week 4, include concepts such as logistic regression and feature scaling. This homework should be submitted for peer review in the assignment titled 4.3 Peer Review: Week 4 Jupyter Notebook. Complete and submit your Jupyter Notebook homework by 11:59pm ET on Sunday. 

In [53]:
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

In [54]:
random_state = 42
classification_reports = []
classification_report_keys = []

## Credit Card

### Import Data

In [55]:
# Import credit card datasaet
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/credit_card_cleaned.csv')
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


### Undersampling due to heavy class imbalance

In [56]:
from collections import Counter


X = df.drop(columns='Class')
y = df['Class']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=random_state)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 284315, 1: 492})
Resampled dataset shape: Counter({0: 4920, 1: 492})


### Split Data

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Logistic Regression
#### (No feature scaling)

In [58]:
model = LogisticRegression(max_iter=10_000, random_state=random_state) # Default max_iter was not converging

In [59]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,10000
,multi_class,'deprecated'


In [60]:
print(model.coef_)

[[-9.60413020e-06  2.96954705e-01  7.63193199e-02 -5.60396363e-02
   8.88890743e-01  1.91699068e-01 -2.55669729e-01 -5.92280204e-02
  -3.96553235e-01 -1.93637194e-03 -6.28836008e-01  1.73629194e-01
  -5.97092753e-01 -3.47359102e-01 -8.47784831e-01 -9.44354525e-02
  -5.38401167e-01 -3.82313888e-01 -5.24724558e-02 -9.22943110e-02
  -4.93752814e-01  9.08236541e-02  5.37836732e-01 -1.45295298e-01
   3.40146345e-01 -1.98914351e-01 -1.13163179e-01 -1.36571414e-01
   2.31397456e-01  2.45177539e-03]]


In [61]:
print(model.n_iter_)

[9062]


In [62]:
y_pred = model.predict(X_test)

In [63]:
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()
classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logsitic Regression')

print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       985
           1       0.95      0.89      0.92        98

    accuracy                           0.99      1083
   macro avg       0.97      0.94      0.95      1083
weighted avg       0.98      0.99      0.99      1083



In [64]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.8496645135231016

### Feature Scaling + Logistic Regression

In [65]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [66]:
model = LogisticRegression(max_iter=10_000, random_state=random_state) # Default max_iter was not converging

In [67]:
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,10000
,multi_class,'deprecated'


In [68]:
y_pred = model.predict(X_test_scaled)

In [69]:
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()
classification_reports.append(df_temp)
classification_report_keys.append('Logsitic Regression with Standard Scaling')

print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       985
           1       0.96      0.89      0.92        98

    accuracy                           0.99      1083
   macro avg       0.97      0.94      0.96      1083
weighted avg       0.99      0.99      0.99      1083



In [70]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.8588898711291159

In [104]:
print(model.n_iter_)

[26]


### Conclusions

In [71]:
pd.concat(classification_reports, keys=classification_report_keys)

precision    recall  \
Baseline Logsitic Regression              0              0.988900  0.994924   
                                          1              0.945652  0.887755   
                                          accuracy       0.985226  0.985226   
                                          macro avg      0.967276  0.941339   
                                          weighted avg   0.984987  0.985226   
Logsitic Regression with Standard Scaling 0              0.988911  0.995939   
                                          1              0.956044  0.887755   
                                          accuracy       0.986150  0.986150   
                                          macro avg      0.972478  0.941847   
                                          weighted avg   0.985937  0.986150   

                                                        f1-score      support  
Baseline Logsitic Regression              0             0.991903   985.000000  
                                          1             0.915789    98.000000  
                                          accuracy      0.985226     0.985226  
                                          macro avg     0.953846  1083.000000  
                                          weighted avg  0.985015  1083.000000  
Logsitic Regression with Standard Scaling 0             0.992413   985.000000  
                                          1             0.920635    98.000000  
                                          accuracy      0.986150     0.986150  
                                          macro avg     0.956524  1083.000000  
                                          weighted avg  0.985918  1083.000000

We can see strong performance in both the baseline and feature scaled models. This was expected behavior to me since all V features (e.g. V1, V2) are preprocessed to be PCA transformed to protect client confidentiality. As a result, these features are already scaled and transformed. The only features affected by scaling are the Amount and Time features which are both numeric. The scaling of these two features could have been the cause of the slight lift in performance (0.953 -> 0.956 macro avg f1-score, 0.9413 -> 0.9418 macro avg recall and finally 0.9672 -> 0.9724 macro avg precision). These differences are very small, but could be impactful when utilizing models on credit card fraud which is extremely high volume transactions.

## IBM

### Import Data

In [72]:
classification_reports = []
classification_report_keys = []

In [73]:
# Import credit card datasaet
df = pd.read_csv('ibm_hi_small_trans_cleaned_semester3.csv')
df

,Is Laundering,Amount_Received_USD,Amount_Paid_USD,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,Receiving Currency_Canadian Dollar,Receiving Currency_Euro,Receiving Currency_Mexican Peso,Receiving Currency_Ruble,Receiving Currency_Rupee,...,Payment Currency_Yen,Payment Currency_Yuan,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,0,3697.340000,3697.340000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,0,0.010000,0.010000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,0,14675.570000,14675.570000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,0,2806.970000,2806.970000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,0,36682.970000,36682.970000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,0,3107.386389,3107.386389,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,0,2168.020464,2168.020464,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,0,100.011894,100.011894,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,0,770.280058,770.280058,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


### Undersampling due to heavy class imbalance

In [74]:
from collections import Counter


X = df.drop(columns='Is Laundering')
y = df['Is Laundering']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=random_state)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 5073168, 1: 5177})
Resampled dataset shape: Counter({0: 5177, 1: 5177})


### Split Data

In [75]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Logistic Regression
#### (No feature scaling)

In [76]:
model = LogisticRegression(random_state=random_state) # Default max_iter was not converging

In [77]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [78]:
y_pred = model.predict(X_test)

In [79]:
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()
classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logsitic Regression')

print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.93      0.22      0.36      1036
           1       0.56      0.98      0.71      1035

    accuracy                           0.60      2071
   macro avg       0.74      0.60      0.54      2071
weighted avg       0.74      0.60      0.54      2071



In [80]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.5574630313713469

### Feature Scaling + Logistic Regression

In [81]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [82]:
model = LogisticRegression(random_state=random_state) # Default max_iter was not converging

In [83]:
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [84]:
y_pred = model.predict(X_test_scaled)

In [85]:
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()
classification_reports.append(df_temp)
classification_report_keys.append('Logsitic Regression with Standard Scaling')

print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.88      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



In [86]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.8432307187875231

### Conclusions

In [87]:
pd.concat(classification_reports, keys=classification_report_keys)

precision    recall  \
Baseline Logsitic Regression              0              0.928000  0.223938   
                                          1              0.558484  0.982609   
                                          accuracy       0.603090  0.603090   
                                          macro avg      0.743242  0.603273   
                                          weighted avg   0.743331  0.603090   
Logsitic Regression with Standard Scaling 0              0.880839  0.891892   
                                          1              0.890411  0.879227   
                                          accuracy       0.885563  0.885563   
                                          macro avg      0.885625  0.885559   
                                          weighted avg   0.885623  0.885563   

                                                        f1-score      support  
Baseline Logsitic Regression              0             0.360809  1036.000000  
                                          1             0.712185  1035.000000  
                                          accuracy      0.603090     0.603090  
                                          macro avg     0.536497  2071.000000  
                                          weighted avg  0.536412  2071.000000  
Logsitic Regression with Standard Scaling 0             0.886331  1036.000000  
                                          1             0.884784  1035.000000  
                                          accuracy      0.885563     0.885563  
                                          macro avg     0.885557  2071.000000  
                                          weighted avg  0.885558  2071.000000

We can see feature scaling played a huge role in this dataset. This could be due to the high scale amount features coefficients. The model could have been under-penalizing them since the coefs would be shrunk due to scale

## PPP

### Import Data

In [88]:
classification_reports = []
classification_report_keys = []

In [89]:
# Import credit card datasaet
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ppp_cleaned.csv')
df

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,0.0,48.0,2.0,24,769358.78,769358.78,11.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12409.01,0,1588291200,1605830400,1608249600
1,0.0,48.0,2.0,24,736927.79,736927.79,11.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10094.90,0,1588291200,1628726400,1632787200
2,0.0,48.0,2.0,24,691355.00,691355.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9218.07,0,1588291200,1612915200,1615939200
3,0.0,48.0,2.0,24,499871.00,499871.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23803.38,0,1588291200,1631232000,1634342400
4,0.0,48.0,2.0,24,367437.00,367437.00,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,14697.48,0,1588291200,1617840000,1629158400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968527,0.0,56.0,2.0,24,150000.00,150000.00,54.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10000.00,0,1585872000,1607472000,1610496000
968528,0.0,56.0,2.0,24,150000.00,150000.00,31.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3452.38,0,1586822400,1604361600,1607385600
968529,1.0,56.0,2.0,60,150000.00,150000.00,54.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,29999.40,0,1613088000,1629158400,1631664000
968530,0.0,56.0,2.0,60,150000.00,150000.00,18.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21428.57,0,1586908800,1645574400,1646697600


### Undersampling due to heavy class imbalance

In [90]:
from collections import Counter


X = df.drop(columns='Fraud')
y = df['Fraud']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=random_state)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 968437, 1: 95})
Resampled dataset shape: Counter({0: 950, 1: 95})


### Split Data

In [91]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Logistic Regression
#### (No feature scaling)

In [92]:
model = LogisticRegression(random_state=random_state) # Default max_iter was not converging

In [93]:
model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [94]:
y_pred = model.predict(X_test)

In [95]:
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()
classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logsitic Regression')

print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      0.89      0.94       190
           1       0.45      0.89      0.60        19

    accuracy                           0.89       209
   macro avg       0.72      0.89      0.77       209
weighted avg       0.94      0.89      0.91       209



In [96]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.40984638630067993

### Feature Scaling + Logistic Regression

In [97]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [98]:
model = LogisticRegression(random_state=random_state) # Default max_iter was not converging

In [99]:
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [100]:
y_pred = model.predict(X_test_scaled)

In [101]:
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()
classification_reports.append(df_temp)
classification_report_keys.append('Logsitic Regression with Standard Scaling')

print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       190
           1       0.72      0.68      0.70        19

    accuracy                           0.95       209
   macro avg       0.85      0.83      0.84       209
weighted avg       0.95      0.95      0.95       209



In [102]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.5228601807549176

### Conclusions

In [103]:
pd.concat(classification_reports, keys=classification_report_keys)

precision    recall  \
Baseline Logsitic Regression              0              0.988304  0.889474   
                                          1              0.447368  0.894737   
                                          accuracy       0.889952  0.889952   
                                          macro avg      0.717836  0.892105   
                                          weighted avg   0.939128  0.889952   
Logsitic Regression with Standard Scaling 0              0.968586  0.973684   
                                          1              0.722222  0.684211   
                                          accuracy       0.947368  0.947368   
                                          macro avg      0.845404  0.828947   
                                          weighted avg   0.946190  0.947368   

                                                        f1-score     support  
Baseline Logsitic Regression              0             0.936288  190.000000  
                                          1             0.596491   19.000000  
                                          accuracy      0.889952    0.889952  
                                          macro avg     0.766390  209.000000  
                                          weighted avg  0.905397  209.000000  
Logsitic Regression with Standard Scaling 0             0.971129  190.000000  
                                          1             0.702703   19.000000  
                                          accuracy      0.947368    0.947368  
                                          macro avg     0.836916  209.000000  
                                          weighted avg  0.946726  209.000000

We see a lift in the performnace with feature scaling. This could be due to large scaled features such as the amount, which end up with naturally smaller coefficients, causing the model to under-penalize them. Scaling guarantees equal shrinkage across all features.